In [14]:
# Match_Soldiers_To_Parents_Partitions.py
# 9/22/2025




import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
import json
import sys
import random
import numpy as np

from dateutil import parser
from dateutil.relativedelta import relativedelta
import ast


def maximize_screen():
    from IPython.display import display,HTML
    display(HTML("<style>:root{--jp-notebook-max-width:100%!important;}</style>"))

def  timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':|',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def  display_dataframe_segments(dataframe_df):
    for i in range(0,len(dataframe_df),50):
        print(dataframe_df.iloc[i:i+50])
        input('')

def clean_dataframe(dataframe_df,column):
    delete_list=['Pvt ','PTE ','Dr. ','Rev ','MAJ ','GEN ','Mrs ','CPT ',',PFC ','SGT ','Col ','LT ','Dr ',\
                ' Sr',' Jr','\.','Minor ','Infant ','Boy ','Girl ','Twins ','PFC','TEC5'\
                'PVT','S Sgt','Capt','Pfc','TSgt','COL','Tec5','TEC4','1','2','III','II','Lt','CPL','SSgt','Sgt',\
                'Staff Sergeant','TEC 5','Maj','GM3','TEC 4','Tec4','RM3','Corp','T 5','MS','  ']
    dataframe_df=dataframe_df.copy()
    for item in delete_list:
        dataframe_df[column]= dataframe_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df      

def discard_empty_sets(dataframe_df,column):
    dataframe_df[column]=dataframe_df[column].apply(
            lambda x:x.discard('') or x if isinstance(x,set) else x)
    return dataframe_df

def  trim_dataframe_set(dataframe_df,column):
    dataframe_df[column]=dataframe_df[column].apply(
            lambda x:{item.strip() for item in x if isinstance(item,str)} if isinstance(x,set) else x)
    return dataframe_df

def match_sets(soldier_match_list_df,search,column_search): 
    search_set= ast.literal_eval(search)
    if isinstance(soldier_match_list_df[column_search].iloc[0], set)==False:
        soldier_match_list_df.loc[:,column_search] = soldier_match_list_df[column_search].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('{') else set()) 
    soldier_match_list_df.loc[:,column_search] = soldier_match_list_df[column_search].apply(
        lambda x: {'nan'} if pd.isna(x) else x)
    soldier_match_list_df = soldier_match_list_df.copy()
    soldier_match_list_df.loc[:,'set_difference']=search_set-soldier_match_list_df[column_search]
    return soldier_match_list_df

def discard_set_one_character(data_set_df,column):
    data_set_df=data_set_df.copy()
    data_set_df[column] = data_set_df[column].apply(
        lambda x: {item for item in ast.literal_eval(x) if len(item) > 2} if isinstance(x, str) else 
              ({item for item in x if len(item) > 1} if isinstance(x, set) else x))
    return data_set_df

def drop_sets_dataframe_duplicates(dataframe_set_df):
    set_columns = [col for col in dataframe_set_df.columns]
    # print(set_columns)
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: tuple(sorted(x)) if isinstance(x, set) else x)   
    dataframe_set_df = dataframe_set_df.drop_duplicates()
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: set(x) if isinstance(x, tuple) else x )
    return dataframe_set_df

def convert_dataframe_to_set(dataframe_df,column,delimiter):
    column_set=column+'_set'
    dataframe_df_split=dataframe_df[column].str.split(delimiter)
    dataframe_df[column_set]=dataframe_df_split.apply(set)
    dataframe_df=discard_empty_sets(dataframe_df,column_set)
    dataframe_df=trim_dataframe_set(dataframe_df,column_set)   
    dataframe_df=discard_set_one_character(dataframe_df,column_set)
    return dataframe_df

def clean_name_set(name_set):
    if not isinstance(name_set, set):
        return set()
    return {
        name.strip()
        for name in name_set
        if isinstance(name, str) and len(name.strip()) > 1}

def clean_name_set_column(name_set):
    if not isinstance(name_set, set):
        return set()
    cleaned_names = set()
    for name in name_set:
        if not isinstance(name, str):
            continue
        # Split name into parts, remove single letters and empty strings
        parts = [p.strip() for p in name.split() if len(p.strip()) > 1]
        if parts:
            cleaned_names.add(' '.join(parts))  # Recombine cleaned parts
    return cleaned_names

def split_set_into_sets(dataframe_set_df,column):
    dataframe_set_df=dataframe_set_df.copy()
    dataframe_set_df[column] = dataframe_set_df[column].apply(
    lambda x: {part.strip() for item in x if isinstance(item, str) for part in item.split() if part.strip()} if isinstance(x, set) else set())
    return dataframe_set_df

def dataframe_contains_match(dataframe_df,search_column,match_column):
    column_out=search_column+'_match'
    dataframe_df[column_out]=dataframe_df.apply(
        lambda row: row[match_column] in row[search_column] if isinstance(row[search_column], str) else False,axis=1)
    return dataframe_df



def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r'[^a-z\s]', '', s)  # keep only letters and spaces
    s = re.sub(r'\s+', ' ', s)      # collapse multiple spaces
    return s.strip()


def haystack_contains_needle(needle_df,haystack_df):
    mask1 = [
    normalize(parent) in normalize(spouse)
    for parent,spouse in zip(
        needle_df,
        haystack_df,  
    )
    ]
    return mask1

def  haystack_contains_needle_fuzzy(dataframe_df,needle,haystack):
    match=haystack+'_match' 
    dataframe_df=dataframe_df.copy()
    dataframe_df[match] = dataframe_df.apply(lambda row: all_parts_in(row[needle], row[haystack]),axis=1)
    mask=dataframe_df[match]
    return dataframe_df, mask

def all_parts_in(needle, haystack):
    if pd.isna(haystack) or pd.isna(needle):
        return None
    needle_parts = needle.split()   # split needle into words
    match_parts = all(part in haystack for part in needle_parts)
    return match_parts

def  match_2_set_columns(searcher_df,searchee_df,searcher_column,searchee_column):
    # searcher_df_split=searcher_df[searcher_column].str.split(' ')
    # searcher_df=searcher_df_split.apply(set)   
    explode_searchee_df=searchee_df.explode(searchee_column)
    print('explode_searchee_df:',explode_searchee_df)
    explode_searchee_df_split=explode_searchee_df[searchee_column].str.split(' ')
    print('\nexplode_searchee_df_split:',explode_searchee_df_split)
    explode_searchee_df=explode_searchee_df_split.apply(set)
    print('\nexplode_searchee_df:',explode_searchee_df)
    print('\nsearcher_df[searcher_column]:',searcher_df[searcher_column])
    
    searcher_matches=searcher_df[searcher_column]-explode_searchee_df
    searcher_matches
    return searcher_matches

def match_soldier_to_child(soldier_df,child_df,soldier_set_column,children_set_column):
    match_set=match_2_set_columns(soldier_df,child_df,soldier_set_column,children_set_column)  
    child_matches_df=pd.DataFrame(match_set,columns=['child_match'])
    match_index = child_matches_df[child_matches_df['child_match']==set()].index
    
    # Initialized Column to Boolian
    soldier_df['children_match']='False'
    soldier_df.loc[match_index, 'children_match'] ='True'
    return soldier_df
    
def single_parent_child_match(parent):
    global soldiers_df,abmc_fag_df
    soldiers_df['Parent_1']=soldiers_df['Parents'].str.split('*').str[0].str.strip()
    soldiers_df['Parent_2']=soldiers_df['Parents'].str.split('*').str[1].str.strip()
    soldiers_df1=soldiers_df[['name_index','soldiers_full_name2','Parent_1','Parent_2','birth_year','soldiers_cemetery','entry_state'\
                              ,'birthStateName','birthCountryName']]

    soldiers_df1.columns=['name_index','soldiers_full_name2','Parent_1','Parent_2','soldiers_birth_year','soldiers_cemetery','entry_state'\
                         ,'soldiers_birth_state','soldiers_birth_country']

    
    abmc_fag_df1=abmc_fag_df[['fullName','Spouses','birthYear','Children','cemeteryName','cemeteryId','prob_jewish','low_grave_count'\
                              ,'memorial_count','birthStateName','birthCountryName','plot']]

    abmc_fag_df1.columns=['full_name','spouses','parent_birth_year','children','cemetery_name','cemetery_id','prob_jewish'\
                              ,'low_grave_count','memorial_count','birth_state','birth_country','plot']
    
    matched_parent_spouse_df11=pd.merge(
        soldiers_df1,
        abmc_fag_df1,
        left_on=[parent],
        right_on=['full_name'],    
        how='inner') 
    matched_parent_spouse_df11['Parent']=matched_parent_spouse_df11[parent]

    print('matched_parent_spouse_df11-Size:',len(matched_parent_spouse_df11))
    matched_parent_spouse_df11=drop_sets_dataframe_duplicates(matched_parent_spouse_df11)
    matched_parent_spouse_df11.reset_index(drop=True, inplace = True)
    
    matched_parent_spouse_df11=clean_dataframe(matched_parent_spouse_df11,'soldiers_full_name2')
    matched_parent_spouse_df11=clean_dataframe(matched_parent_spouse_df11,'children')
    
    needle='soldiers_full_name2'
    haystack='children'
    dataframe_df=matched_parent_spouse_df11
    matched_parent_spouse_df11, mask=haystack_contains_needle_fuzzy(dataframe_df,needle,haystack)
    
    matched_out_df11c=pd.merge(
        soldiers_df1,
        matched_parent_spouse_df11,
        left_on=[ 'name_index'],
        right_on=['name_index'],
        how='inner')
    print('matched_out_df11c-Size:',len(matched_out_df11c))
    matched_out_df11c=drop_sets_dataframe_duplicates(matched_out_df11c)
    matched_out_df11c.reset_index(drop=True, inplace = True)
    print('matched_out_df11c-Size:',len(matched_out_df11c))
    
    matched_out_df11c=drop_sets_dataframe_duplicates(matched_out_df11c)
    matched_out_df11c.reset_index(drop=True, inplace = True)
    print('matched_out_df11c-Size:',len(matched_out_df11c))
    
    reorder_format=['name_index','soldiers_full_name2_x','Parent','spouses','Parent_1_x', 'Parent_2_x','full_name','children_match','prob_jewish','low_grave_count', 'soldiers_birth_year_x', 'parent_birth_year','birth_state'\
      ,'entry_state_x','birth_country','children','cemetery_name','cemetery_id','memorial_count','soldiers_cemetery_x','plot', 'soldiers_birth_state_x', 'soldiers_birth_country_x']
    
    rename_format=['name_index','soldiers_full_name','parent','spouse','Parent_1', 'Parent_2','full_name','children_match','prob_jewish','low_grave_count', 'soldiers_birth_year', 'parent_birth_year','birth_state'\
                    ,'entry_state','birth_country','children','cemetery_name','cemetery_id','memorial_count','soldiers_cemetery','plot','soldiers_birth_state','soldiers_birth_country']
    matched_out_df11c=matched_out_df11c[reorder_format]
    matched_out_df11c.columns=rename_format
    matched_out_df11c['soldiers_birth_year'] = pd.to_numeric(matched_out_df11c['soldiers_birth_year'], errors='coerce').astype('Int64')
    matched_out_df11c['parent_birth_year'] = pd.to_numeric(matched_out_df11c['parent_birth_year'], errors='coerce').astype('Int64')
    
    matched_out_df11c['Age']=matched_out_df11c['soldiers_birth_year']-matched_out_df11c['parent_birth_year'].astype('Int64')
    return matched_out_df11c
    
def  search_dataframe(dataframe,column,search_text):
    dataframe_extract=dataframe[dataframe[column]==search_text]
    return dataframe_extract

def clean_dataframe(dataframe_df,column):
    delete_list=['Pvt ','PTE ','Dr. ','Rev ','MAJ ','GEN ','Mrs ','CPT ',',PFC ','SGT ','Col ','LT ','Dr ',\
                ' Sr',' Jr','\.','Minor ','Infant ','Boy ','Girl ','Twins ','PFC ','TEC5 ','\[','\]',"\'",'\"',
                'PVT ','S Sgt ','Capt ','Pfc ','TSgt ','COL ','Tec5 ','TEC4 ','1','2','III','II','Lt','CPL','SSgt','Sgt',\
                'Staff Sergeant ','TEC 5 ','Maj ','GM3 ','TEC 4 ','Tec4 ','RM3 ','Corp ','T 5 ','MS ','  ']
    dataframe_df=dataframe_df.copy()
    for item in delete_list:
        dataframe_df[column]= dataframe_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df 

def time_lapse(tic):
    toc=datetime.datetime.now()
    processing_time=toc-tic
    processing_time_seconds=int(processing_time.total_seconds())
    processing_time_minutes=processing_time_seconds/60
    return f" {processing_time_minutes:.1f}"

def time_now():
    time_now=datetime.datetime.now()  
    return time_now
        

blank_line='                                                                                                                                                                                           '

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Match Soldiers to Parents By Partitions'
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')

DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileInputSoldier="ABMC_FAG_Soldiers_Master_List"
fileInputSoD="star_david_20250915_df"
fileInputCemetery="FAG_Cemeteries_Affiliation_20250928" 
    
fileInSold=DirMain+"0-Master Worksheets\\"+fileInputSoldier+".csv"
fileInCemetery=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Inputs\\"+fileInputCemetery+".csv"
fileInSoD=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Inputs\\"+fileInputSoD+".csv"

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% Read In %%%%%%%%%%%%%%
soldiers_df=pd.read_csv(fileInSold,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
print('• soldiers_df * Download Complete * Size:',len(soldiers_df))
cemetery_df=pd.read_csv(fileInCemetery,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
print('• cemetery_df * Download Complete * Size:',len(cemetery_df))
star_david_df=pd.read_csv(fileInSoD)
print('• Star of David * Download Complete * Size:',len(star_david_df),'\n')
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% Read In %%%%%%%%%%%%%%

na_index=soldiers_df[soldiers_df['middle_name'].isna()].index
soldiers_df.loc[na_index,'middle_name']=''
soldiers_df['soldiers_full_name2']=soldiers_df['first_name']+' '+soldiers_df['middle_name']+' '+soldiers_df['last_name']

group_A=['Relatives_A0','FAG_Relationship_Relatives_A_','Relatives_A',1,3]
group_B=['Relatives_B','FAG_Relationship_Missing_Name_Graves_POJ_','Relatives_B',1,48]
group_C=['Relatives_C','FAG_Relationship_Names_Remain_Graves_','Relatives_C',0,99]
partition_df=pd.DataFrame([group_B,group_C,group_A])
# partition_df=pd.DataFrame([group_A])
# partition_df=pd.DataFrame([group_A1,group_A2,group_A3,group_A4])
partition_df_list=['directory','format','label','range_start','range_end']
partition_df.columns=[partition_df_list]

for index, row in partition_df.iterrows():
    subdirectory=row['directory']
    file_format=row['format']
    label=row['label']
    partition_start=row['range_start']
    partition_end=row['range_end']
    fileOutRelatives=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\Soldier_Parents_Match_"+label+"_"+timestamp+".csv"
    fileMissingCemeteries=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\Missing_Cemeteries_"+label+".csv"
    start=partition_start
    end=partition_end
    abmc_fag_df=pd.DataFrame([])
    missing_cemetery_id_df=pd.DataFrame([])
    tic=time_now()
    for file_number in range(start,end):
        file_number_str='0'+str(file_number)
        file_number_str=file_number_str[-2:]
        fileInRelatives=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Inputs\\"+subdirectory+"\\"+file_format+file_number_str+".csv"
        # abmc_fag_df=pd.read_csv(fileInRelatives,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
        abmc_fag_partitions_df=pd.read_csv(fileInRelatives,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
    
        abmc_fag_partitions_df = abmc_fag_partitions_df.dropna(subset=['Children'])               

        abmc_fag_partitions_df['cemeteryId'] = pd.to_numeric(abmc_fag_partitions_df['cemeteryId'], errors='coerce').astype('Int64')       
        cemetery_df_correlation=pd.merge(
        abmc_fag_partitions_df,
        cemetery_df[['cemetery_id','prob_jewish','low_grave_count','memorial_count']],
        left_on=['cemeteryId'],
        right_on=['cemetery_id'],
        how='left')

        abmc_fag_partitions_df=cemetery_df_correlation.copy()
        abmc_fag_partitions_df.drop(columns=['cemetery_id'], inplace=True)

        missing_partition_cemetery_id=cemetery_df_correlation[cemetery_df_correlation['cemetery_id'].isna()]
        missing_cemetery_id_df=pd.concat([missing_cemetery_id_df,missing_partition_cemetery_id])
        
        abmc_fag_df=pd.concat([abmc_fag_df,abmc_fag_partitions_df])
        abmc_fag_df=drop_sets_dataframe_duplicates(abmc_fag_df)
        abmc_fag_df.reset_index(drop=True, inplace = True)
        # print(f" {blank_line}", end='\r')
        # print(f"Group: {label} • Uploading File {file_format+file_number_str} •  Number of Partitions: {end} • Record Count: {len(abmc_fag_partitions_df):,}/ {len(abmc_fag_df):,}", end='\r') 
        print(f"Group: {label} • Uploading File {file_format+file_number_str} •  Number of Partitions: {end} • Record Count: {len(abmc_fag_partitions_df):,}/ {len(abmc_fag_df):,}") 
    if 'plot' not in list(abmc_fag_df):abmc_fag_df['plot']='No Plot Data'
    process_time= time_lapse(tic)
    print('\n',f" Group Type Uploaded: {label} • Records: {len(abmc_fag_df):,} • Process Time: {process_time} Minutes")

   # $$$$$$$$$$$ Clean databases fields $$$$$$$$$$$$$$$$$$$$$

    tic=time_now()
    
    print('* Clean Databases Fields Start *')
    column_set=['soldiers_full_name','Parents', 'Spouses', 'Siblings', 'Children']
    for column in column_set:
        soldiers_df=clean_dataframe(soldiers_df,column)
    
    column_set=['fullName', 'Spouses', 'Siblings','Children']
    for column in column_set:
        abmc_fag_df=clean_dataframe(abmc_fag_df,column)
    
    process_time= time_lapse(tic)
    print('* Clean Databases Fields Complete * Process Time:',process_time)
    
    FAG_Mereged_length=len(soldiers_df)
    Relative_Pre_length=len(abmc_fag_df)
    
    file_soldiers_df=DirMain+"Affilation_Analysis\\Inputs\\soldiers_df.csv"
    file_abmc_fag_df=DirMain+"Affilation_Analysis\\Inputs\\abmc_fag_df_"+label+".csv"
    file_star_david_df=DirMain+"Affilation_Analysis\\Inputs\\star_david_20250915_df.csv"
    
    soldiers_df.to_csv(file_soldiers_df,mode="w",index=False,header=True)
    abmc_fag_df.to_csv(file_abmc_fag_df,mode="w",index=False,header=True)
    star_david_df.to_csv(file_star_david_df,mode="w",index=False,header=True)
   
    end_time=datetime.datetime.now()
    processing_time=end_time-start_time
    processing_time_seconds=int(processing_time.total_seconds())
    print(f"** Downloading Complete **** • FAG_Mereged Length: {FAG_Mereged_length} •  Relative_Pre Length: {Relative_Pre_length}  ••  Processing Time: {processing_time_seconds} Sec")

    #$$$$$$$$$$$$$$$$$$$$$ Merge each soldier $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
    # # OKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOK
    skip=1
    if skip==0:
        tic=time_now()
            
        file_soldiers_df=DirMain+"Affilation_Analysis\\Inputs\\soldiers_df.csv"
        file_abmc_fag_df=DirMain+"Affilation_Analysis\\Inputs\\abmc_fag_df_B.csv"
        file_star_david_df=DirMain+"Affilation_Analysis\\Inputs\\star_david_20250915_df.csv"
        
        soldiers_df=pd.read_csv(file_soldiers_df,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
        print('• soldiers_df * Download Complete *')
        
        abmc_fag_df=pd.read_csv(file_abmc_fag_df,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
        print('• abmc_fag_df * Download Complete *')
        
        star_david_df=pd.read_csv(file_star_david_df)
        process_time= time_lapse(tic)
        print('• Star of David * Download Complete *')
        print('Process Time:',process_time)
    # &&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&
    
    parent='Parent_1'
    matched_out_df1=single_parent_child_match(parent)
    parent='Parent_2'
    matched_out_df2=single_parent_child_match(parent)
    matched_out_df1=drop_sets_dataframe_duplicates(matched_out_df1)
    matched_out_df1.reset_index(drop=True, inplace = True)
    matched_out_df2=drop_sets_dataframe_duplicates(matched_out_df2)
    matched_out_df2.reset_index(drop=True, inplace = True)

    file_matched_out_df=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\\matched_out_df1_"+file_format+file_number_str+".csv"
    matched_out_df1.to_csv(file_matched_out_df,mode="w",index=False,header=True)
    file_matched_out_df=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\\matched_out_df2_"+file_format+file_number_str+".csv"
    matched_out_df2.to_csv(file_matched_out_df,mode="w",index=False,header=True)    
    input('')
    
    matched_out_df12_inner_df=pd.merge(
        matched_out_df1,
        matched_out_df2,
        left_on=[ 'name_index'],
        right_on=['name_index'],
        how='inner')
    
    matched_out_df12_inner_df=matched_out_df12_inner_df[matched_out_df12_inner_df['spouse_y'].notna()]
    matched_out_df12_inner_df=matched_out_df12_inner_df[matched_out_df12_inner_df['spouse_x'].notna()]
    matched_out_df12_inner_df.reset_index(drop=True, inplace = True)
    
    needle='parent_x'
    haystack='spouse_y'
    dataframe_df=matched_out_df12_inner_df
    dataframe_df, mask=haystack_contains_needle_fuzzy(dataframe_df,needle,haystack)
    
    needle='parent_y'
    haystack='spouse_x'
    dataframe_df=dataframe_df
    dataframe_df, mask=haystack_contains_needle_fuzzy(dataframe_df,needle,haystack)
    dataframe_df['cemetery_match']=dataframe_df['cemetery_name_x']==dataframe_df['cemetery_name_y']
    
    matched_out_df12_inner_df=drop_sets_dataframe_duplicates(dataframe_df)
    matched_out_df12_inner_df.reset_index(drop=True, inplace = True)
    
    matched_soldier_affiliation_sod_df=pd.merge(
        matched_out_df12_inner_df,
        star_david_df,
        left_on=[ 'name_index'],
        right_on=['Name_Index'],
        how='left')
    
    matched_soldier_affiliation_sod_df['Star_of_David'] = matched_soldier_affiliation_sod_df['Name_Index'].fillna(False)
    matched_soldier_affiliation_sod_df['Star_of_David']=matched_soldier_affiliation_sod_df['Star_of_David']!=False
    matched_soldier_affiliation_sod_df.drop(columns=['Name_Index'], inplace=True)
    matched_soldier_affiliation_sod_df.drop_duplicates(inplace=True)
    
    reorder_format=['name_index','soldiers_full_name_x','parent_x','spouse_y','parent_y','spouse_x','spouse_y_match','spouse_x_match'\
                    ,'cemetery_match','children_match_x','children_match_y','prob_jewish_x','prob_jewish_y','Star_of_David','cemetery_name_x'\
                    ,'cemetery_name_y','plot_x','plot_y','cemetery_id_x','cemetery_id_y','children_x','children_y','soldiers_cemetery_x'\
                    ,'soldiers_birth_state_x','soldiers_birth_country_x','entry_state_x','birth_state_x','birth_country_x','birth_state_y'\
                    ,'birth_country_y','memorial_count_x','memorial_count_y']
    
    rename_format=['name_index','soldiers_full_name','parent_1','spouse_1','parent_2','spouse_2','p1-s2_match','p2-s1_match','p1_p2_cemetery_match'\
                   ,'children_match_p1','children_match_p2','prob_jewish_1','prob_jewish_2','Star_of_David','cemetery_name_1','cemetery_name_2'\
                   ,'plot_1','plot_2','cemetery_id_1','cemetery_id_2','children_1','children_2','soldiers_cemetery','soldiers_birth_state'\
                   ,'soldiers_birth_country','entry_state','birth_state_1','birth_country_1','birth_state_2','birth_country_2','memorial_count_1'\
                   ,'memorial_count_2']
    
    matched_soldier_affiliation_sod_df=matched_soldier_affiliation_sod_df[reorder_format]
    matched_soldier_affiliation_sod_df.columns=rename_format
    matched_soldier_affiliation_sod_df = matched_soldier_affiliation_sod_df.sort_values(by='name_index',ascending=True)
    matched_soldier_affiliation_sod_df.drop_duplicates(inplace=True)
    matched_soldier_affiliation_sod_df.reset_index(drop=True, inplace = True)
    
    matched_soldier_affiliation_sod_df.to_csv(fileOutRelatives,mode="w",index=False,header=True)	
    missing_cemetery_id_df.to_csv(fileMissingCemeteries,mode="w",index=False,header=True)
    process_time= time_lapse(tic)
    print("\n")
    print(subdirectory+' - UploadedTo:',fileOutRelatives,'Number of Rows:',len(matched_soldier_affiliation_sod_df),' • Process Time:',process_time,'\n\n')  
    
    end_time=datetime.datetime.now()
    processing_time=end_time-start_time
    processing_time_seconds=int(processing_time.total_seconds())
    print(f"**** {process_name}   Processing Time: {processing_time_seconds/60:6.1f} Minutes")    


**Starting ***  Match Soldiers to Parents By Partitions  *** , Time: 2025-09-30 11:40:44.973031  **

• soldiers_df * Download Complete * Size: 68801
• cemetery_df * Download Complete * Size: 251399
• Star of David * Download Complete * Size: 1645 

Group: Relatives_B • Uploading File FAG_Relationship_Missing_Name_Graves_POJ_01 •  Number of Partitions: 48 • Record Count: 14,077/ 14,077
Group: Relatives_B • Uploading File FAG_Relationship_Missing_Name_Graves_POJ_02 •  Number of Partitions: 48 • Record Count: 42,393/ 56,470
Group: Relatives_B • Uploading File FAG_Relationship_Missing_Name_Graves_POJ_03 •  Number of Partitions: 48 • Record Count: 79,716/ 136,183
Group: Relatives_B • Uploading File FAG_Relationship_Missing_Name_Graves_POJ_04 •  Number of Partitions: 48 • Record Count: 19,851/ 155,530
Group: Relatives_B • Uploading File FAG_Relationship_Missing_Name_Graves_POJ_05 •  Number of Partitions: 48 • Record Count: 8,189/ 163,337
Group: Relatives_B • Uploading File FAG_Relationship_M



Relatives_B - UploadedTo: C:\Users\jrozm\OneDrive\Benjamin\0-Master Worksheets\Soldier-Parent Processing\Outputs\Soldier_Parents_Match_Relatives_B_202509302.csv Number of Rows: 6093  • Process Time:  45.6 


**** Match Soldiers to Parents By Partitions   Processing Time:   54.5 Minutes
Group: Relatives_C • Uploading File FAG_Relationship_Names_Remain_Graves_00 •  Number of Partitions: 99 • Record Count: 15,572/ 15,572
Group: Relatives_C • Uploading File FAG_Relationship_Names_Remain_Graves_01 •  Number of Partitions: 99 • Record Count: 51,297/ 66,869
Group: Relatives_C • Uploading File FAG_Relationship_Names_Remain_Graves_02 •  Number of Partitions: 99 • Record Count: 49,545/ 116,413
Group: Relatives_C • Uploading File FAG_Relationship_Names_Remain_Graves_03 •  Number of Partitions: 99 • Record Count: 48,436/ 164,848
Group: Relatives_C • Uploading File FAG_Relationship_Names_Remain_Graves_04 •  Number of Partitions: 99 • Record Count: 48,522/ 213,370
Group: Relatives_C • Uploading Fi



Relatives_C - UploadedTo: C:\Users\jrozm\OneDrive\Benjamin\0-Master Worksheets\Soldier-Parent Processing\Outputs\Soldier_Parents_Match_Relatives_C_202509302.csv Number of Rows: 9704  • Process Time:  31.7 


**** Match Soldiers to Parents By Partitions   Processing Time:  136.2 Minutes
Group: Relatives_A • Uploading File FAG_Relationship_Relatives_A_01 •  Number of Partitions: 3 • Record Count: 2,343,133/ 2,343,133
Group: Relatives_A • Uploading File FAG_Relationship_Relatives_A_02 •  Number of Partitions: 3 • Record Count: 2,321,454/ 4,664,572

  Group Type Uploaded: Relatives_A • Records: 4,664,572 • Process Time:  2.0 Minutes
* Clean Databases Fields Start *
* Clean Databases Fields Complete * Process Time:  12.8
** Downloading Complete **** • FAG_Mereged Length: 68801 •  Relative_Pre Length: 4664572  ••  Processing Time: 9107 Sec
matched_parent_spouse_df11-Size: 21173
matched_out_df11c-Size: 22743
matched_out_df11c-Size: 22743
matched_out_df11c-Size: 22743
matched_parent_spouse_d



Relatives_A0 - UploadedTo: C:\Users\jrozm\OneDrive\Benjamin\0-Master Worksheets\Soldier-Parent Processing\Outputs\Soldier_Parents_Match_Relatives_A_202509302.csv Number of Rows: 60690  • Process Time:  37.6 


**** Match Soldiers to Parents By Partitions   Processing Time:  175.8 Minutes


In [ ]:
soldiers_df=pd.read_csv(fileInSold,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
na_index=soldiers_df[soldiers_df['middle_name'].isna()].index
soldiers_df.loc[na_index,'middle_name']=''
soldiers_df['soldiers_full_name2']=soldiers_df['first_name']+' '+soldiers_df['middle_name']+' '+soldiers_df['last_name']
column_set=['soldiers_full_name','Parents', 'Spouses', 'Siblings', 'Children']
for column in column_set:
    soldiers_df=clean_dataframe(soldiers_df,column)
    
parent='Parent_1'
matched_out_df1=single_parent_child_match(parent)
parent='Parent_2'
matched_out_df2=single_parent_child_match(parent)
matched_out_df1=drop_sets_dataframe_duplicates(matched_out_df1)
matched_out_df1.reset_index(drop=True, inplace = True)
matched_out_df2=drop_sets_dataframe_duplicates(matched_out_df2)
matched_out_df2.reset_index(drop=True, inplace = True)

file_matched_out_df=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\\matched_out_df1.csv"
matched_out_df1.to_csv(file_matched_out_df,mode="w",index=False,header=True)
file_matched_out_df=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\\matched_out_df2.csv"
matched_out_df2.to_csv(file_matched_out_df,mode="w",index=False,header=True)
print('$$$ Process Completed $$$')

In [12]:
list(soldiers_df)

['ï»¿name_index',
 'last_name',
 'first_name',
 'middle_name',
 'soldiers_full_name',
 'Unnamed: 5',
 'birthDate',
 'birth_year',
 'deathDate',
 'birth_index',
 'death_index',
 'entry_state',
 'birthCityName',
 'birthStateName',
 'birthCountryName',
 'soldiers_cemetery',
 'links',
 'plot',
 'Parents',
 'Spouses',
 'Siblings',
 'Children',
 'soldiers_full_name2',
 'Parent_1',
 'Parent_2']

In [ ]:
# fileND_SectionsIn=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Inputs\\Jewish_Keywords.csv"
# jewish_keywords_df=pd.read_csv(fileND_SectionsIn)
# print('• jewish_keywords_df * Download Complete *, Size:',len(jewish_keywords_df))

fileOutRelatives=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\Soldier_Parents_Match_Relatives_B_202509262.csv"
matched_soldier_affiliation_sod_df=pd.read_csv(fileOutRelatives,encoding='ISO-8859-1')
print('• matched_soldier_affiliation_sod_df * Download Complete, Size: *',len(matched_soldier_affiliation_sod_df))

matched_soldier_affiliation_sod_df_test=matched_soldier_affiliation_sod_df.copy()

skip=1
if skip==0:
    jewish_keywords_merge_df = matched_soldier_affiliation_sod_df_test.merge(
        jewish_keywords_df1,
        left_on=["cemetery_id_1"],
        right_on=["cemetery_id"],
        how="left"
    )
    
    jewish_keywords_plot_df1, mask =haystack_contains_needle_fuzzy(jewish_keywords_merge_df1,'keyword','plot_1')
    print(jewish_keywords_plot_df)
    
    file_jewish_keywords_plot=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\jewish_keywords_plot_df1.csv"
    jewish_keywords_plot_df.to_csv(file_jewish_keywords_plot,mode="w",index=False,header=True)	
    input('')

jewish_keywords_merge_df2 = matched_soldier_affiliation_sod_df_test.merge(
    jewish_keywords_df,
    left_on=["cemetery_id_2"],
    right_on=["cemetery_id"],
    how="left"
)

jewish_keywords_plot_df2, mask =haystack_contains_needle_fuzzy(jewish_keywords_merge_df2,'keyword','plot_2')
print(jewish_keywords_plot_df2)

file_jewish_keywords_plot=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Outputs\\jewish_keywords_plot_df2.csv"
jewish_keywords_plot_df.to_csv(file_jewish_keywords_plot,mode="w",index=False,header=True)	
input('')



drop_index=jewish_keywords_merge_df[jewish_keywords_merge_df['plot'].isna()].index
jewish_keywords_merge_df = jewish_keywords_merge_df.drop(index=drop_index)
mask_clean = mask.drop(index=drop_index)
jewish_keywords_plot_inner_df=jewish_keywords_merge_df[mask_clean]
graves_df_01.loc[jewish_keywords_plot_inner_df["graves_index"], "tags"] = jewish_keywords_plot_inner_df["tags"].values

# fileSectionsOut=DirMain+"Affilation_Analysis\\Outputs\\jewish_keywords_plot_inner_df.csv"
# jewish_keywords_plot_inner_df.to_csv(fileSectionsOut,mode="w",index=False,header=True)



graves_plot_extracted_df = graves_df_01.copy()
graves_plot_extracted_df["normalized"] = graves_plot_extracted_df["plot"].apply(normalize_plot_letter)


# print(graves_plot_extracted_df["plot"].apply(extract_section_code))
graves_plot_extracted_df["letter_section"]=graves_plot_extracted_df["plot"].apply(extract_section_code)
graves_plot_extracted_df["section"] = graves_plot_extracted_df["normalized"].apply(extract_section)

# Merge all tags in prioritized order
for col in ["letter_section", "section", "tags"]:
    mask = graves_plot_extracted_df[col].notna()
    graves_plot_extracted_df.loc[mask, "Merged"] = graves_plot_extracted_df.loc[mask, col]

print(graves_plot_extracted_df["Merged"])

In [ ]:
len(matched_soldier_affiliation_sod_df_test)

In [ ]:
column_set=['fullName', 'Spouses', 'Siblings','Children']
for column in column_set:
    abmc_fag_df=clean_dataframe(abmc_fag_df,column)

process_time= time_lapse(tic)
print('* Clean Databases Fields Complete * Process Time:',process_time)



FAG_Mereged_length=len(soldiers_df)
Relative_Pre_length=len(abmc_fag_df)

file_soldiers_df=DirMain+"Affilation_Analysis\\Inputs\\soldiers_df.csv"
file_abmc_fag_df=DirMain+"Affilation_Analysis\\Inputs\\abmc_fag_df_"+subdirectory+".csv"
file_star_david_df=DirMain+"Affilation_Analysis\\Inputs\\star_david_20250915_df.csv"

soldiers_df.to_csv(file_soldiers_df,mode="w",index=False,header=True)
abmc_fag_df.to_csv(file_abmc_fag_df,mode="w",index=False,header=True)
star_david_df.to_csv(file_star_david_df,mode="w",index=False,header=True)

end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"** Downloading Complete **** • FAG_Mereged Length: {FAG_Mereged_length} •  Relative_Pre Length: {Relative_Pre_length}  ••  Processing Time: {processing_time_seconds} Sec")

#$$$$$$$$$$$$$$$$$$$$$ Merge each soldier $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
# # OKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOK
skip=1
if skip==0:
    tic=time_now()
        
    file_soldiers_df=DirMain+"Affilation_Analysis\\Inputs\\soldiers_df.csv"
    file_abmc_fag_df=DirMain+"Affilation_Analysis\\Inputs\\abmc_fag_df_B.csv"
    file_star_david_df=DirMain+"Affilation_Analysis\\Inputs\\star_david_20250915_df.csv"
    
    soldiers_df=pd.read_csv(file_soldiers_df,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
    print('• soldiers_df * Download Complete *')
    
    abmc_fag_df=pd.read_csv(file_abmc_fag_df,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
    print('• abmc_fag_df * Download Complete *')
    
    star_david_df=pd.read_csv(file_star_david_df)
    process_time= time_lapse(tic)
    print('• Star of David * Download Complete *')
    print('Process Time:',process_time)
# &&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&

parent='Parent_1'
matched_out_df1=single_parent_child_match(parent)
parent='Parent_2'
matched_out_df2=single_parent_child_match(parent)
matched_out_df1=drop_sets_dataframe_duplicates(matched_out_df1)
matched_out_df1.reset_index(drop=True, inplace = True)
matched_out_df2=drop_sets_dataframe_duplicates(matched_out_df2)
matched_out_df2.reset_index(drop=True, inplace = True)

matched_out_df12_inner_df=pd.merge(
    matched_out_df1,
    matched_out_df2,
    left_on=[ 'name_index'],
    right_on=['name_index'],
    how='inner')

matched_out_df12_inner_df=matched_out_df12_inner_df[matched_out_df12_inner_df['spouse_y'].notna()]
matched_out_df12_inner_df=matched_out_df12_inner_df[matched_out_df12_inner_df['spouse_x'].notna()]
matched_out_df12_inner_df.reset_index(drop=True, inplace = True)

needle='parent_x'
haystack='spouse_y'
dataframe_df=matched_out_df12_inner_df
dataframe_df, mask=haystack_contains_needle_fuzzy(dataframe_df,needle,haystack)

needle='parent_y'
haystack='spouse_x'
dataframe_df=dataframe_df
dataframe_df, mask=haystack_contains_needle_fuzzy(dataframe_df,needle,haystack)
dataframe_df['cemetery_match']=dataframe_df['cemetery_name_x']==dataframe_df['cemetery_name_y']

matched_out_df12_inner_df=drop_sets_dataframe_duplicates(dataframe_df)
matched_out_df12_inner_df.reset_index(drop=True, inplace = True)

matched_soldier_affiliation_sod_df=pd.merge(
    matched_out_df12_inner_df,
    star_david_df,
    left_on=[ 'name_index'],
    right_on=['Name_Index'],
    how='left')

matched_soldier_affiliation_sod_df['Star_of_David'] = matched_soldier_affiliation_sod_df['Name_Index'].fillna(False)
matched_soldier_affiliation_sod_df['Star_of_David']=matched_soldier_affiliation_sod_df['Star_of_David']!=False
matched_soldier_affiliation_sod_df.drop(columns=['Name_Index'], inplace=True)
matched_soldier_affiliation_sod_df.drop_duplicates(inplace=True)

reorder_format=['name_index','soldiers_full_name_x','parent_x','spouse_y','parent_y','spouse_x','spouse_y_match','spouse_x_match'\
                ,'cemetery_match','children_match_x','children_match_y','prob_jewish_x','prob_jewish_y','Star_of_David','cemetery_name_x'\
                ,'cemetery_name_y','plot_x','plot_y','cemetery_id_x','cemetery_id_y','children_x','children_y','soldiers_cemetery_x'\
                ,'soldiers_birth_state_x','soldiers_birth_country_x','entry_state_x','birth_state_x','birth_country_x','birth_state_y'\
                ,'birth_country_y','memorial_count_x','memorial_count_y']

rename_format=['name_index','soldiers_full_name','parent_1','spouse_1','parent_2','spouse_2','p1-s2_match','p2-s1_match','p1_p2_cemetery_match'\
               ,'children_match_p1','children_match_p2','prob_jewish_1','prob_jewish_2','Star_of_David','cemetery_name_1','cemetery_name_2'\
               ,'plot_1','plot_2','cemetery_id_1','cemetery_id_2','children_1','children_2','soldiers_cemetery','soldiers_birth_state'\
               ,'soldiers_birth_country','entry_state','birth_state_1','birth_country_1','birth_state_2','birth_country_2','memorial_count_1'\
               ,'memorial_count_2']

matched_soldier_affiliation_sod_df=matched_soldier_affiliation_sod_df[reorder_format]
matched_soldier_affiliation_sod_df.columns=rename_format
matched_soldier_affiliation_sod_df = matched_soldier_affiliation_sod_df.sort_values(by='name_index',ascending=True)
matched_soldier_affiliation_sod_df.drop_duplicates(inplace=True)
matched_soldier_affiliation_sod_df.reset_index(drop=True, inplace = True)

matched_soldier_affiliation_sod_df.to_csv(fileOutRelatives,mode="w",index=False,header=True)	
missing_cemetery_id_df.to_csv(fileMissingCemeteries,mode="w",index=False,header=True)
process_time= time_lapse(tic)
print("\n")
print(subdirectory+' - UploadedTo:',fileOutRelatives,'Number of Rows:',len(matched_soldier_affiliation_sod_df),' • Process Time:',process_time)  

end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")    

In [ ]:
list(abmc_fag_partitions_df)